In [ ]:
import os
from pathlib import Path

from bots.utils.tactical.annotate_possible_orders import annotate_possible_orders

from bots.utils.tactical.estimate_bundle_score import estimate_bundle_score

from spec.bundle_scoring_harness import (
    list_saved_game_phases,
    load_bundle_search_inputs,
    load_game_at_phase,
    run_bundle_search_from_saved_game,
)
from bots.utils.tactical import select_best_order_bundle


def get_location(possible_orders, name):
    return next((obj for obj in possible_orders if obj["location"] == name), None)

print(f"os.getcwd()={os.getcwd()}")
GAME_PATH = Path(os.getcwd()).resolve() / 'spec' / 'game_data' / '3x3a.json'
PHASE = 'F1902M'
POWER = 'AUSTRIA'
phases = list_saved_game_phases(GAME_PATH)
print(f"{phases}")
game = load_game_at_phase(GAME_PATH, PHASE)
search_inputs = load_bundle_search_inputs(GAME_PATH, PHASE, POWER)
print()
possible_orders = search_inputs.possible_orders
all_orders = set()
for orders_dict in possible_orders:
    all_orders.update(orders_dict['orders'])

print(all_orders)
#attacker_abc = next((obj for obj in possible_orders if obj["location"] == "ABC"), None)
abc = get_location(possible_orders, "ABC")
acb = get_location(possible_orders, "ACB")

#supporter_acb = 
attacking_order = "A ABC - ABB"
supporting_order = "A ACB S A ABC - ABB"
assert attacking_order in abc['orders']
assert supporting_order in acb['orders']
assert attacking_order in all_orders
assert supporting_order in all_orders

annotations = annotate_possible_orders(
    power_name=search_inputs.power_name,
    possible_orders=search_inputs.possible_orders,
    units_by_power=search_inputs.units_by_power,
    centers_by_power=search_inputs.centers_by_power,
    loc_abut=search_inputs.loc_abut,
)
annotation_by_order = {str(entry['order']): entry for entry in annotations}

best_bundle = run_bundle_search_from_saved_game(GAME_PATH, PHASE, POWER, beam_width=64)
recommended_orders = best_bundle['recommended_orders']

dangling_support_orders = [
    'A ABC S A ACB - ABB',
    'A ACB S A ABC - ABB',
    'A ACC S A ABC - ACB',
]
direct_attack_orders = [
    'A ABC - ABB',
    'A ACB S A ABC - ABB',
    'A ACC H',
]
super_support_orders = [
    'A ABC - ABB',
    'A ACB S A ABC - ABB',
    'A AAC S A ABC - ABB'
]

order_list = {
    "dangling_support_orders": dangling_support_orders,
    "direct_attack_orders": direct_attack_orders,
    "super_support_orders": super_support_orders,
    "highest_bundle_orders": recommended_orders
}



for key in order_list:
    display("----------")
    display(f"order_name: {key}")
    display(f"orders: {order_list[key]}")
    orders = order_list[key]
    score, breakdown, _ = estimate_bundle_score(
        power_name=search_inputs.power_name,
        orders=orders,
        annotation_by_order=annotation_by_order,
        units_by_power=search_inputs.units_by_power,
        centers_by_power=search_inputs.centers_by_power,
        loc_abut=search_inputs.loc_abut,
        supply_centers=search_inputs.supply_centers,
    )
    display(score)
    display(breakdown)

display(recommended_orders)